# Hands-on: AI Recommendation System untuk Membantu UMKM

**Studi kasus:** merekomendasikan solusi ketika UMKM menghadapi masalah bisnis  
**Platform:** Google Colab | **Level:** Pemula | **Estimasi:** 60–90 menit

## Tujuan pembelajaran
Setelah menyelesaikan notebook ini, peserta dapat:
1. Mengubah deskripsi masalah UMKM menjadi embedding menggunakan pretrained model.
2. Menghitung cosine similarity dan menghasilkan Top-K rekomendasi.
3. Memfilter solusi berdasarkan kategori, perkiraan biaya, dan cakupan wilayah.
4. Menjelaskan keterbatasan sistem dan pentingnya memvalidasi rekomendasi.

> **Catatan penting:** katalog solusi di notebook ini adalah data contoh/sintetis untuk pembelajaran, bukan daftar program bantuan yang sudah diverifikasi. Sebelum digunakan oleh UMKM sungguhan, ganti atau lengkapi katalog dengan informasi dari sumber resmi dan periksa kelayakan, biaya, wilayah, serta tanggal pembaruannya.

## 1. Gambaran alur sistem

1. Pelaku UMKM menjelaskan masalahnya, misalnya: "penjualan menurun dan pelanggan jarang kembali".
2. Sistem membandingkan deskripsi masalah dengan deskripsi solusi dalam katalog.
3. Pretrained model mengubah teks menjadi embedding numerik.
4. Cosine similarity memberi skor kemiripan teks.
5. Sistem mengurutkan solusi, lalu menerapkan filter sederhana seperti anggaran dan wilayah.

**Istilah penting**
- **Pretrained model:** model yang sudah dilatih sebelumnya.
- **Embedding:** representasi numerik dari teks.
- **Cosine similarity:** ukuran kemiripan antara dua vektor; skor lebih tinggi berarti lebih mirip menurut model, bukan jaminan solusi akan berhasil.
- **Content-based recommendation:** rekomendasi berdasarkan isi/deskripsi item, bukan perilaku pengguna lain.

**Batasan:** notebook ini adalah prototipe content-based recommender. Ia tidak membuktikan bahwa solusi efektif, tidak memberikan nasihat keuangan profesional, dan tidak otomatis memverifikasi ketersediaan program.

## 2. Instalasi library\nJalankan sel berikut di Google Colab.

In [ ]:
%pip -q install sentence-transformers scikit-learn pandas

## 3. Import library

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

## 4. Membuat katalog solusi UMKM

Dataset mini ini dibuat langsung di notebook supaya peserta bisa fokus pada konsep. Deskripsi dibuat cukup spesifik agar model punya konteks untuk dicocokkan.

Kolom `biaya_perkiraan_rp` adalah ilustrasi untuk latihan filter, **bukan harga pasar atau penawaran nyata**. `wilayah` dan `provider` juga perlu diverifikasi bila katalog diganti dengan program nyata.

In [ ]:
solutions = pd.DataFrame([
    {
        "solution_id": "S001",
        "nama_solusi": "Audit pemasaran sederhana",
        "kategori": "Pemasaran",
        "deskripsi": "Memeriksa target pelanggan, pesan promosi, kanal pemasaran, dan alasan penjualan menurun. Susun eksperimen promosi kecil dan ukur hasilnya.",
        "target_usaha": "Semua sektor",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Semua wilayah",
        "tingkat_kesulitan": "Mudah",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S002",
        "nama_solusi": "Katalog digital dan konten produk",
        "kategori": "Digitalisasi",
        "deskripsi": "Membuat katalog produk digital dengan foto, harga, deskripsi, kontak pemesanan, serta jadwal unggahan media sosial.",
        "target_usaha": "Semua sektor",
        "biaya_perkiraan_rp": 100000,
        "wilayah": "Semua wilayah",
        "tingkat_kesulitan": "Mudah",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S003",
        "nama_solusi": "Pencatatan arus kas",
        "kategori": "Keuangan",
        "deskripsi": "Mencatat pemasukan, pengeluaran, utang, piutang, dan saldo kas secara rutin untuk mengetahui kondisi keuangan usaha.",
        "target_usaha": "Semua sektor",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Semua wilayah",
        "tingkat_kesulitan": "Mudah",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S004",
        "nama_solusi": "Perencanaan stok dan persediaan",
        "kategori": "Operasional",
        "deskripsi": "Mencatat stok masuk dan keluar, mengenali barang yang paling laku, menetapkan stok minimum, dan mengurangi stok menumpuk atau kehabisan.",
        "target_usaha": "Retail dan kuliner",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Semua wilayah",
        "tingkat_kesulitan": "Sedang",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S005",
        "nama_solusi": "Eksplorasi pilihan pembiayaan",
        "kategori": "Pembiayaan",
        "deskripsi": "Menyusun kebutuhan modal, tujuan penggunaan dana, kemampuan membayar, dan membandingkan persyaratan pembiayaan dari lembaga resmi.",
        "target_usaha": "Usaha yang membutuhkan modal",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Perlu cek penyedia",
        "tingkat_kesulitan": "Sedang",
        "provider": "Contoh panduan; bukan penawaran pinjaman",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S006",
        "nama_solusi": "Perbaikan kemasan dan identitas merek",
        "kategori": "Produk",
        "deskripsi": "Mengevaluasi kemasan, label, informasi produk, identitas visual, dan kejelasan manfaat agar produk lebih mudah dikenali pelanggan.",
        "target_usaha": "Kuliner dan kerajinan",
        "biaya_perkiraan_rp": 250000,
        "wilayah": "Semua wilayah",
        "tingkat_kesulitan": "Sedang",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S007",
        "nama_solusi": "Riset pelanggan sederhana",
        "kategori": "Pelanggan",
        "deskripsi": "Melakukan wawancara singkat atau survei pelanggan untuk memahami kebutuhan, keluhan, alasan membeli, dan alasan tidak membeli kembali.",
        "target_usaha": "Semua sektor",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Semua wilayah",
        "tingkat_kesulitan": "Mudah",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S008",
        "nama_solusi": "Perhitungan harga pokok dan harga jual",
        "kategori": "Keuangan",
        "deskripsi": "Menghitung biaya bahan baku, tenaga kerja, kemasan, biaya operasional, margin, dan harga jual agar tidak menjual di bawah biaya.",
        "target_usaha": "Kuliner, produksi, dan kerajinan",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Semua wilayah",
        "tingkat_kesulitan": "Sedang",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S009",
        "nama_solusi": "Pemetaan pemasok dan biaya logistik",
        "kategori": "Rantai pasok",
        "deskripsi": "Membandingkan pemasok, kualitas bahan, jumlah minimum pesanan, waktu pengiriman, ongkos logistik, dan risiko keterlambatan.",
        "target_usaha": "Semua sektor",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Perlu cek lokasi",
        "tingkat_kesulitan": "Sedang",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S010",
        "nama_solusi": "Pencarian pelatihan dan pendampingan resmi",
        "kategori": "Pelatihan",
        "deskripsi": "Mencari pelatihan kewirausahaan, pemasaran digital, pembukuan, keamanan pangan, sertifikasi, atau pendampingan dari lembaga pemerintah dan mitra yang terverifikasi.",
        "target_usaha": "Semua sektor",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Perlu cek program lokal",
        "tingkat_kesulitan": "Mudah",
        "provider": "Katalog program perlu dilengkapi",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S011",
        "nama_solusi": "Evaluasi kanal penjualan",
        "kategori": "Penjualan",
        "deskripsi": "Membandingkan penjualan melalui toko fisik, pesan langsung, marketplace, reseller, pameran, dan kemitraan berdasarkan biaya serta jangkauan pelanggan.",
        "target_usaha": "Semua sektor",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Semua wilayah",
        "tingkat_kesulitan": "Sedang",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    },
    {
        "solution_id": "S012",
        "nama_solusi": "Standarisasi proses produksi",
        "kategori": "Operasional",
        "deskripsi": "Membuat langkah kerja standar, memantau kualitas produk, mengurangi bahan terbuang, dan menjaga konsistensi produksi.",
        "target_usaha": "Kuliner, produksi, dan kerajinan",
        "biaya_perkiraan_rp": 0,
        "wilayah": "Semua wilayah",
        "tingkat_kesulitan": "Sedang",
        "provider": "Contoh solusi mandiri",
        "resource_url": "",
        "status_verifikasi": "Contoh sintetis"
    }
])

display(solutions[["solution_id", "nama_solusi", "kategori", "biaya_perkiraan_rp", "wilayah", "status_verifikasi"]])
print("Jumlah solusi:", len(solutions))

### Catatan diskusi
- Apakah setiap masalah bisnis hanya memiliki satu solusi?
- Bagaimana jika satu solusi tidak cocok untuk wilayah atau kemampuan UMKM?
- Informasi apa yang perlu diverifikasi sebelum merekomendasikan program bantuan?
- Mengapa rekomendasi tidak boleh hanya didasarkan pada skor kemiripan?

## 5. Memuat pretrained model multilingual

Kita menggunakan `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. Model ini mendukung banyak bahasa, sehingga lebih sesuai untuk eksperimen teks bahasa Indonesia dibanding model yang terutama berorientasi bahasa Inggris.

Model tetap perlu diuji dengan contoh lokal. Hasil embedding bukan pemahaman sempurna atas konteks bisnis, bahasa daerah, atau kelayakan program.

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
print("Model berhasil dimuat:", MODEL_NAME)

## 6. Mengubah deskripsi solusi menjadi embedding

Kita menghitung embedding katalog satu kali. Jika katalog diubah, jalankan kembali sel ini sebelum melakukan rekomendasi.

In [ ]:
solution_texts = (
    solutions["nama_solusi"].fillna("") + ". " +
    solutions["kategori"].fillna("") + ". " +
    solutions["deskripsi"].fillna("") + ". Sasaran: " +
    solutions["target_usaha"].fillna("")
).tolist()

solution_embeddings = model.encode(
    solution_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Bentuk matriks embedding:", solution_embeddings.shape)
print("Jumlah solusi:", len(solutions))
print("Dimensi embedding per solusi:", solution_embeddings.shape[1])

**Cara membaca output:** jika bentuknya `(12, 384)`, artinya terdapat 12 solusi dan setiap solusi direpresentasikan oleh 384 angka. Angka-angka tersebut adalah representasi teks, bukan skor kualitas solusi.

## 7. Membuat fungsi rekomendasi

Fungsi berikut:
1. menerima deskripsi masalah UMKM,
2. membuat embedding masalah,
3. menghitung cosine similarity,
4. mengurutkan solusi berdasarkan skor,
5. menerapkan filter biaya maksimum dan kategori bila diminta.

**Catatan:** filter biaya menggunakan angka ilustratif pada dataset ini. Untuk data nyata, pastikan arti biaya jelas (misalnya biaya solusi, bukan jumlah modal yang dibutuhkan) dan gunakan nilai yang sudah diverifikasi.

In [ ]:
def recommend_solutions(
    problem,
    top_k=5,
    max_cost=None,
    category=None
):
    if not isinstance(problem, str) or not problem.strip():
        raise ValueError("Deskripsi masalah harus berupa teks yang tidak kosong.")
    if top_k < 1:
        raise ValueError("top_k harus minimal 1.")

    query_embedding = model.encode(
        [problem],
        normalize_embeddings=True
    )
    scores = cosine_similarity(query_embedding, solution_embeddings)[0]

    results = solutions.copy()
    results["skor_kemiripan"] = scores

    if max_cost is not None:
        results = results[results["biaya_perkiraan_rp"] <= max_cost]

    if category is not None and category != "Semua kategori":
        results = results[results["kategori"] == category]

    results = results.sort_values("skor_kemiripan", ascending=False).head(top_k).copy()
    results["skor_kemiripan"] = results["skor_kemiripan"].round(3)

    columns = [
        "solution_id", "nama_solusi", "kategori", "deskripsi",
        "biaya_perkiraan_rp", "wilayah", "provider",
        "resource_url", "status_verifikasi", "skor_kemiripan"
    ]
    return results[columns].reset_index(drop=True)

### Memahami kode
- `model.encode([problem])`: mengubah masalah menjadi embedding.
- `cosine_similarity(...)`: menghitung skor terhadap semua solusi.
- `sort_values(...)`: mengurutkan solusi dari skor tertinggi.
- `max_cost`: filter sederhana untuk membatasi biaya ilustratif.
- `category`: filter kategori solusi.
- `top_k`: jumlah rekomendasi teratas.

Skor kemiripan bukan probabilitas keberhasilan dan bukan akurasi model.

## 8. Uji rekomendasi pertama\nContoh masalah: penjualan turun dan pelanggan jarang membeli kembali.

In [ ]:
problem = "Penjualan usaha makanan saya menurun. Pelanggan jarang kembali membeli dan promosi di media sosial belum efektif."
recommend_solutions(problem, top_k=5)

## 9. Eksperimen dengan masalah UMKM yang berbeda

Perhatikan apakah rekomendasi berubah ketika konteks masalah berbeda.

In [ ]:
test_problems = [
    "Saya sering kehabisan stok barang yang laku, tetapi barang lain justru menumpuk.",
    "Saya tidak tahu apakah usaha saya untung karena semua uang tercampur dengan uang pribadi.",
    "Biaya bahan baku dan pengiriman mahal, apalagi jika barang harus dikirim ke luar daerah.",
    "Produk kerajinan saya bagus tetapi kemasannya kurang menarik dan sulit dikenal pembeli.",
    "Saya membutuhkan modal tambahan dan ingin mengetahui pilihan pembiayaan yang aman."
]

for p in test_problems:
    print("\nMASALAH:", p)
    display(recommend_solutions(p, top_k=3)[
        ["nama_solusi", "kategori", "skor_kemiripan", "status_verifikasi"]
    ])

### Pertanyaan refleksi
1. Apakah rekomendasi teratas masuk akal?
2. Apakah ada solusi yang relevan tetapi tidak muncul di Top-3?
3. Apakah deskripsi solusi yang terlalu umum memengaruhi hasil?
4. Bagaimana cara meminta umpan balik dari pelaku UMKM untuk mengevaluasi hasil?

## 10. Coba filter biaya dan kategori

Filter ini adalah aturan bisnis sederhana di atas hasil similarity. Filter tidak membuat model lebih pintar, tetapi dapat membantu menyaring opsi.

In [ ]:
problem = "Saya ingin meningkatkan penjualan tetapi anggaran promosi sangat terbatas."

print("Semua kategori, biaya ilustratif maksimal Rp100.000")
display(recommend_solutions(problem, top_k=5, max_cost=100_000))

print("\nHanya kategori Keuangan")
display(recommend_solutions(
    "Saya kesulitan mengetahui biaya dan keuntungan usaha.",
    top_k=5,
    category="Keuangan"
))

## 11. Tampilkan semua skor untuk audit ranking

Melihat semua hasil membantu peserta memeriksa apakah solusi yang tidak relevan mendapat skor tinggi. Jangan memilih solusi hanya karena skornya paling tinggi; baca deskripsi, persyaratan, biaya, dan status verifikasinya.

In [ ]:
def score_all_solutions(problem):
    query_embedding = model.encode([problem], normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, solution_embeddings)[0]
    result = solutions.copy()
    result["skor_kemiripan"] = scores.round(3)
    return result.sort_values("skor_kemiripan", ascending=False).reset_index(drop=True)

score_all_solutions("Modal usaha terbatas dan saya perlu mengatur arus kas.")

## 12. Menghubungkan dengan data publik

Notebook ini sengaja memakai katalog mini agar bisa langsung dijalankan. Untuk versi proyek nyata, peserta dapat mengembangkan dua sumber data:

**A. Data profil dan kondisi UMKM**
- [Satu Data Indonesia — pencarian dataset UMKM](https://data.go.id/dataset?q=usaha+mikro)
- [World Bank Enterprise Surveys — data perusahaan](https://www.enterprisesurveys.org/en/data)
- [World Bank Enterprise Surveys — Indonesia 2023](https://microdata.worldbank.org/index.php/catalog/6449)

**B. Katalog solusi/program**
- Kumpulkan program pelatihan, pendampingan, sertifikasi, atau pembiayaan dari situs resmi pemerintah daerah, kementerian, inkubator, dan lembaga penyedia.
- Simpan nama program, penyedia, deskripsi, sasaran, syarat, wilayah, biaya, tautan resmi, tanggal verifikasi, dan status aktif.
- Periksa lisensi/ketentuan penggunaan setiap sumber sebelum mengunduh atau memublikasikan datanya.

Sumber publik di atas tidak otomatis menyediakan pasangan label masalah-solusi. Data profil perusahaan juga bukan pengganti katalog solusi. Siswa perlu melakukan pemetaan, kurasi, dan validasi.

## 13. Ekspor katalog contoh menjadi CSV

CSV ini memudahkan peserta mengganti katalog sintetis dengan data yang sudah dikurasi. File hasil ekspor tetap berisi contoh sintetis sampai peserta menggantinya.

In [ ]:
csv_path = "katalog_solusi_umkm_contoh.csv"
solutions.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("CSV tersimpan:", csv_path)

## 14. Tantangan peserta

**Tantangan A — Tambahkan solusi**
- Tambahkan minimal 5 solusi yang relevan.
- Buat deskripsi yang spesifik dan tidak menjanjikan hasil yang pasti.
- Jalankan ulang sel embedding setelah katalog diubah.

**Tantangan B — Buat katalog lokal**
- Pilih satu sektor, misalnya kuliner, kerajinan, pertanian, atau perdagangan.
- Tambahkan konteks wilayah dan hambatan logistik jika relevan.
- Untuk Papua atau wilayah lain, pastikan program dan penyedianya memang tersedia di wilayah tersebut.

**Tantangan C — Evaluasi**
- Buat 10 contoh masalah dan minta pengajar/mentor memberi label solusi relevan.
- Bandingkan Top-3 model dengan label manusia.
- Diskusikan kasus ketika solusi yang paling mirip secara teks tidak layak dijalankan.

**Tantangan D — Pengembangan**
- Baca katalog dari CSV.
- Tambahkan filter jenis usaha dan lokasi.
- Buat UI sederhana di Streamlit.
- Simpan katalog terverifikasi di database bila proyek berkembang.

## 15. Troubleshooting
- **Model lama dimuat:** unduhan pertama membutuhkan internet dan mungkin beberapa menit.
- **RAM/runtime penuh:** mulai dari katalog kecil, restart runtime, lalu jalankan sel dari atas.
- **Dataset diubah tetapi hasil tidak berubah:** hitung ulang `solution_embeddings`.
- **Rekomendasi bahasa Indonesia kurang relevan:** bandingkan hasil pada contoh berlabel; pertimbangkan model multilingual lain.
- **Tidak ada hasil setelah filter:** naikkan `max_cost`, pilih `Semua kategori`, atau periksa kategori yang tersedia.
- **Solusi program nyata muncul tetapi ternyata tidak tersedia:** sistem membutuhkan data program terverifikasi dan tanggal pembaruan, bukan hanya model embedding.

## 16. Ringkasan

Alur hands-on:

**Deskripsi masalah UMKM → embedding pretrained → cosine similarity → filter sederhana → ranking Top-K**

Kita tidak melakukan fine-tuning atau training model dari nol. Kita menggunakan pretrained model sebagai pembuat embedding, kemudian menulis logika rekomendasi.

### Referensi teknis
- Model card: https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- Dokumentasi Sentence Transformers: https://www.sbert.net/
- Dokumentasi cosine similarity: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html